# 데이터 마트 전처리 (4종)

이 노트북은 팀 공유용 마트 4종(`mart_point_payment_event`, `mart_safety_event`, `mart_question_exposure`, `mart_user_network_snapshot`)의 **최종 전처리 로직**만 정리한 버전입니다. 각 단계에서 발견한 이슈와 처리 근거는 `전처리_작업_노트(mart_point_payment_event, mart_safety_event, mart_question_exposure, mart_user_network_snapshot)`(Notion) 문서를 참고하세요.

> `mart_funnel_session`은 별도로 진행 중이라 이번 공유에서 제외했습니다.

**실행 전 준비**: 팀 구글 드라이브에 업로드된 5개 마트 CSV 원본을 로컬 `csv_export/` 폴더에 받아두세요.

## 0. 환경 설정

In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

csv_dir = os.path.join(os.getcwd(), "csv_export")


## 1. mart_point_payment_event (포인트·결제 마트)

In [2]:
mart_point_payment = pd.read_csv(os.path.join(csv_dir, "mart_point_payment_event.csv"))
mart_point_payment.shape


/var/folders/x5/h1lsczws5m7f12d8vz60hpbc0000gn/T/ipykernel_3490/2053977852.py:1: DtypeWarning: Columns (0: session_id, 1: product_id, 2: phone_type, 3: is_success) have mixed types. Specify dtype option on import or set low_memory=False.
  mart_point_payment = pd.read_csv(os.path.join(csv_dir, "mart_point_payment_event.csv"))


(2476068, 17)

### 1-1. dtype 정리

In [3]:
mart_point_payment['event_at'] = pd.to_datetime(mart_point_payment['event_at'], errors='coerce')
mart_point_payment['is_success'] = mart_point_payment['is_success'].astype('boolean')

int_cols = ['user_id', 'user_question_record_id', 'ping_question_id', 'ping_has_read']
for col in int_cols:
    mart_point_payment[col] = mart_point_payment[col].astype('Int64')


### 1-2. 결측치 처리

hackle 소스 이벤트(SHOP_VIEW/PRODUCT_CLICK/PURCHASE_COMPLETE_EVENT)에서 세션-유저 매핑에 실패한 `user_id` 결측 68건은 팀 논의 후 **행 삭제**로 결정했습니다.

In [4]:
before = len(mart_point_payment)
mart_point_payment = mart_point_payment.dropna(subset=['user_id']).reset_index(drop=True)
after = len(mart_point_payment)
print(f"삭제된 행: {before - after}건 (68건이어야 정상)")


삭제된 행: 68건 (68건이어야 정상)


### 1-3. 처리하지 않고 유지하기로 한 항목 (참고)

- `PAYMENT_FAIL`의 `product_id` 결측(107건, 전부 iOS)은 iOS 결제 실패 로깅의 구조적 한계로 추정되며 복구 불가하여 결측인 채로 유지합니다.
- `is_outlier` 플래그는 전체 데이터에서 전부 `False`로 확인되었습니다 (이상 거래 미탐지, 참고용으로만 기록).

In [5]:
mart_point_payment.info()


<class 'pandas.DataFrame'>
RangeIndex: 2476000 entries, 0 to 2475999
Data columns (total 17 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   value_event_id           str           
 1   event_type               str           
 2   event_at                 datetime64[us]
 3   user_id                  Int64         
 4   session_id               str           
 5   identity_status          str           
 6   point_delta              float64       
 7   point_amount_abs         float64       
 8   product_id               str           
 9   phone_type               str           
 10  is_success               boolean       
 11  user_question_record_id  Int64         
 12  ping_question_id         Int64         
 13  ping_has_read            Int64         
 14  ping_answer_status       str           
 15  source_table             str           
 16  is_outlier               bool          
dtypes: Int64(4), bool(1), boolean(1), date

In [6]:
mart_point_payment.to_csv(os.path.join(csv_dir, "mart_point_payment_event_clean.csv"), index=False)


## 2. mart_safety_event (안전·신고 마트)

In [7]:
mart_safety = pd.read_csv(os.path.join(csv_dir, "mart_safety_event.csv"))
mart_safety.shape


/var/folders/x5/h1lsczws5m7f12d8vz60hpbc0000gn/T/ipykernel_3490/3668599510.py:1: DtypeWarning: Columns (0: is_safety_related, 1: target_region) have mixed types. Specify dtype option on import or set low_memory=False.
  mart_safety = pd.read_csv(os.path.join(csv_dir, "mart_safety_event.csv"))


(71283, 16)

### 2-1. dtype 정리

In [8]:
mart_safety['event_at'] = pd.to_datetime(mart_safety['event_at'], errors='coerce')
mart_safety['is_safety_related'] = mart_safety['is_safety_related'].astype('boolean')

int_cols = ['target_user_id', 'question_id', 'question_piece_id',
            'user_question_record_id', 'actor_school_id', 'target_school_id']
for col in int_cols:
    mart_safety[col] = mart_safety[col].astype('Int64')


### 2-2. 자기 자신 신고/차단 케이스 삭제

`actor_user_id == target_user_id`인 36건을 발견했습니다 (USER_BLOCK 33건, TIMELINE_REPORT 3건). 2023-05-21 하루에 20건(55%)이 몰려있어 특정 시점의 시스템 버그로 추정되며, `target_user_id`를 신뢰할 수 없는 값으로 판단해 팀 논의 후 **행 삭제**로 결정했습니다.

In [9]:
before2 = len(mart_safety)
self_target_mask = (
    (mart_safety['actor_user_id'] == mart_safety['target_user_id']) &
    mart_safety['target_user_id'].notna()
)
mart_safety = mart_safety[~self_target_mask].reset_index(drop=True)
after2 = len(mart_safety)
print(f"삭제된 행: {before2 - after2}건 (36건이어야 정상)")


삭제된 행: 36건 (36건이어야 정상)


### 2-3. 처리하지 않고 유지하기로 한 항목 (참고)

`reason_category` 6종 중 "불쾌·민감", "긍정 피드백"이 이 데이터 기간에는 전혀 나타나지 않았습니다. "기타"로 분류된 건들에서 관련 키워드를 직접 검색해봤지만 매칭되는 것이 없어, 분류 로직 오류가 아니라 해당 기간에 그런 사유 자체가 없었던 것으로 결론지었습니다 (별도 조치 없음).

In [10]:
mart_safety.info()


<class 'pandas.DataFrame'>
RangeIndex: 71247 entries, 0 to 71246
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   safety_event_id          71247 non-null  str           
 1   safety_event_type        71247 non-null  str           
 2   event_at                 71247 non-null  datetime64[us]
 3   actor_user_id            71247 non-null  int64         
 4   target_user_id           19823 non-null  Int64         
 5   question_id              51593 non-null  Int64         
 6   question_piece_id        169 non-null    Int64         
 7   user_question_record_id  374 non-null    Int64         
 8   reason_raw               71078 non-null  str           
 9   reason_category          71078 non-null  str           
 10  is_safety_related        71078 non-null  boolean       
 11  actor_school_id          71247 non-null  Int64         
 12  target_school_id         19823 non-null  In

In [11]:
mart_safety.to_csv(os.path.join(csv_dir, "mart_safety_event_clean.csv"), index=False)


## 3. mart_question_exposure (질문 노출 마트)

In [12]:
mart_question = pd.read_csv(os.path.join(csv_dir, "mart_question_exposure.csv"))
mart_question.shape


(1583840, 35)

### 3-1. dtype 정리

In [13]:
datetime_cols = ['opening_time', 'question_set_created_at', 'exposure_at']
for col in datetime_cols:
    mart_question[col] = pd.to_datetime(mart_question[col], errors='coerce')

mart_question['exposure_date'] = pd.to_datetime(mart_question['exposure_date'], errors='coerce').dt.date

flag_cols = ['piece_exists_flag', 'has_vote_record', 'set_completed_flag',
             'voted_and_skipped_flag', 'voted_without_record_flag', 'voter_user_mismatch_flag']
for col in flag_cols:
    mart_question[col] = mart_question[col].astype('boolean')


### 3-2. 논리적 흐름 오류(오픈 시각 < 생성 시각) 값 처리

질문세트의 `opening_time`이 `question_set_created_at`보다 빠른 비정상 케이스가 679개 세트(10행씩 반복되어 6,790행, 전체의 0.43%)에서 발견되었습니다. 세트 자체는 유지하되, 신뢰할 수 없는 `opening_time` 값만 결측 처리하고 `has_invalid_time_order` 플래그로 표시합니다.

In [14]:
mart_question['has_invalid_time_order'] = (
    mart_question['opening_time'].notna() &
    mart_question['question_set_created_at'].notna() &
    (mart_question['opening_time'] < mart_question['question_set_created_at'])
)

invalid_count = mart_question['has_invalid_time_order'].sum()
mart_question.loc[mart_question['has_invalid_time_order'], 'opening_time'] = pd.NaT
mart_question['has_invalid_time_order'] = mart_question['has_invalid_time_order'].astype('boolean')
print(f"opening_time을 결측 처리한 행: {invalid_count}건 (6,790건이어야 정상)")


opening_time을 결측 처리한 행: 6790건 (6,790건이어야 정상)


### 3-3. 처리하지 않고 유지하기로 한 항목 (참고)

`voted_and_skipped_flag`(1,127건)와 `voted_without_record_flag`(1,129건)는 대부분(1,098건) 서로 겹치는 동일한 원인(`raw_is_voted=1`인데 실제 투표 기록 없음)이었습니다. 마트의 `final_outcome` 컬럼이 이미 이런 경우를 `SKIPPED_ONLY`로 올바르게 재분류해두었기 때문에 **행을 삭제하지 않고 유지**합니다.

⚠️ **분석 시 주의**: 질문 완료/스킵 여부를 판단할 때는 `raw_is_voted` / `raw_is_skipped`가 아니라 반드시 **`final_outcome`을 기준으로 사용**하세요.

⚠️ **집계 시 주의**: `set_completed_flag`는 세트당 10행에 동일한 값이 반복되므로, 세트 단위로 집계할 때는 `question_set_id` 기준 중복 제거 후 계산해야 합니다 (그대로 SUM하면 10배로 부풀려짐).

In [15]:
mart_question.info()


<class 'pandas.DataFrame'>
RangeIndex: 1583840 entries, 0 to 1583839
Data columns (total 36 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   question_set_id            1583840 non-null  int64         
 1   question_position          1583840 non-null  int64         
 2   question_piece_id          1583840 non-null  int64         
 3   user_id                    1583840 non-null  int64         
 4   gender                     1583840 non-null  str           
 5   group_id                   1583840 non-null  int64         
 6   grade                      1583840 non-null  int64         
 7   class_num                  1583840 non-null  int64         
 8   school_id                  1583840 non-null  int64         
 9   school_type                1583840 non-null  str           
 10  question_set_status        1583840 non-null  str           
 11  opening_time               1577050 non-null  dat

In [16]:
mart_question.to_csv(os.path.join(csv_dir, "mart_question_exposure_clean.csv"), index=False)


## 4. mart_user_network_snapshot (유저 네트워크 마트)

In [17]:
mart_network = pd.read_csv(os.path.join(csv_dir, "mart_user_network_snapshot.csv"))
mart_network.shape


(677085, 54)

### 4-1. dtype 정리

In [18]:
datetime_cols = ['mart_built_at', 'user_created_at']
for col in datetime_cols:
    mart_network[col] = pd.to_datetime(mart_network[col], errors='coerce')

bool_cols = ['is_superuser', 'is_staff', 'is_isolated_current_snapshot',
             'is_isolated_valid_network', 'is_inbound_only_user', 'has_accepted_request_partner']
for col in bool_cols:
    mart_network[col] = mart_network[col].astype('boolean')

int_cols = ['group_id', 'grade', 'class_num', 'school_id']
for col in int_cols:
    mart_network[col] = mart_network[col].astype('Int64')


### 4-2. 검증 결과 (참고)

- 원본 데이터(`accounts_user.friend_id_list`) 대조 감사: 무작위 5명 + orphan/중복 존재 9명 샘플 전원에서 raw/valid/orphan/중복/mutual 값이 100% 일치
- 전체 677,085명 기준 정합성 공식(`mutual+one_way=degree`, `valid_outbound<=raw`, `raw=valid+orphan+dup`) 전부 0건 위반
- `mart_built_at < user_created_at`인 논리 오류 0건

→ 추가 수정 없이 그대로 사용 가능한 상태로 확인되었습니다.

### 4-3. school_address 관련 정정

명세서에는 "준식별 정보, 공개 저장소 업로드 금지"로 기재되어 있었으나, 실제 값을 직접 확인한 결과 "경기도 군포시", "서울특별시 은평구"처럼 **시/도+시/군/구 수준의 지역명**일 뿐 도로명·번지 등 전체 주소가 아님을 확인했습니다. 공개 공유해도 무방한 것으로 판단합니다.

In [19]:
mart_network['school_address'].dropna().unique()[:10]


<StringArray>
[     '경기도 군포시', '충청남도 천안시 동남구', '충청남도 천안시 서북구',      '경기도 화성시',
      '경기도 의왕시',     '충청북도 충주시',    '부산광역시 강서구',    '서울특별시 은평구',
    '서울특별시 강남구',  '경기도 안산시 단원구']
Length: 10, dtype: str

In [20]:
mart_network.info()


<class 'pandas.DataFrame'>
RangeIndex: 677085 entries, 0 to 677084
Data columns (total 54 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   mart_built_at                      677085 non-null  datetime64[us]
 1   user_id                            677085 non-null  int64         
 2   gender                             677083 non-null  str           
 3   user_created_at                    677085 non-null  datetime64[us]
 4   is_superuser                       677085 non-null  boolean       
 5   is_staff                           677085 non-null  boolean       
 6   ban_status                         677085 non-null  str           
 7   group_id                           677082 non-null  Int64         
 8   grade                              677082 non-null  Int64         
 9   class_num                          677082 non-null  Int64         
 10  school_id                      

In [21]:
mart_network.to_csv(os.path.join(csv_dir, "mart_user_network_snapshot_clean.csv"), index=False)


## 5. 요약

| 마트 | 최종 행 수 | 삭제 | 비고 |
|---|---|---|---|
| mart_point_payment_event | 원본 - 68건 | user_id 결측 68건 | is_outlier 전부 False (참고용) |
| mart_safety_event | 원본 - 36건 | 자기 자신 신고/차단 36건 | is_self_target 컬럼은 삭제 후 제거됨 |
| mart_question_exposure | 원본과 동일 | 없음 | opening_time 679세트분 결측 처리, final_outcome 사용 권장 |
| mart_user_network_snapshot | 원본과 동일 | 없음 | 원본 대조 감사 포함 전 항목 검증 통과 |